In [0]:
import sys
sys.path.append('./..')

from config.tables.table_config import TABLE_CONFIG
df = spark.table(TABLE_CONFIG.DATA_PROCESSING)

In [0]:
df.select('account_id', 'age', 'gender', 'credit_card_limit', 'registered_on_days')

In [0]:
from pyspark.sql import functions as F

transaction_features = (
    df
    .groupBy("account_id")
    .agg(
        F.count(F.when(F.col("event") == "transaction", 1)).alias("transaction_frequency"),
        F.coalesce(
            F.sum(
                F.when(F.col("event") == "transaction", F.col("amount"))
                .otherwise(0)
            ),
            F.lit(0)
        ).alias("total_spend"),
        F.avg(
            F.when(F.col("event") == "transaction", F.col("amount"))
        ).alias("avg_transaction_value"),
        F.stddev(
            F.when(F.col("event") == "transaction", F.col("amount"))
        ).alias("transaction_value_std"),
        F.max(
            F.when(F.col("event") == "transaction", F.col("amount"))
        ).alias("max_transaction"),
        F.min(
            F.when(F.col("event") == "transaction", F.col("amount"))
        ).alias("min_transaction")
    )
)

transaction_features.display()

In [0]:
from pyspark.sql import functions as F

offer_features = (
    df
    .groupBy("account_id")
    .agg(
        F.count(
            F.when(F.col("event") == "offer received", 1)
        ).alias("offers_received_count"),

        F.count(
            F.when(F.col("event") == "offer viewed", 1)
        ).alias("offers_viewed_count"),

        F.count(
            F.when(F.col("event") == "offer completed", 1)
        ).alias("offers_completed_count")
    )
    .withColumn(
        "view_rate",
        F.when(
            F.col("offers_received_count") > 0,
            F.col("offers_viewed_count") / F.col("offers_received_count")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "completion_rate_after_view",
        F.when(
            F.col("offers_viewed_count") > 0,
            F.col("offers_completed_count") / F.col("offers_viewed_count")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "overall_completion_rate",
        F.when(
            F.col("offers_received_count") > 0,
            F.col("offers_completed_count") / F.col("offers_received_count")
        ).otherwise(F.lit(0.0))
    )
)

offer_features.display()

In [0]:
from pyspark.sql import functions as F

reward_features = (
    df
    .groupBy("account_id")
    .agg(
        F.coalesce(
            F.sum(
                F.when(F.col("event") == "offer completed", F.col("reward"))
                .otherwise(0)
            ),
            F.lit(0)
        ).alias("total_rewards_earned"),

        F.avg(
            F.when(F.col("event") == "offer completed", F.col("reward"))
        ).alias("avg_reward_per_completion")
    )
)

reward_features.display()

In [0]:
time_features = (
    df
    .groupBy("account_id")
    .agg(
        F.max(
            F.when(
                F.col("event") == "transaction",
                F.col("time_since_test_start")
            )
        ).alias("last_transaction_time"),

        F.min(
            F.when(
                F.col("event") == "transaction",
                F.col("time_since_test_start")
            )
        ).alias("first_transaction_time"),

        (
            F.max("time_since_test_start")
            - F.min("time_since_test_start")
        ).alias("customer_activity_span"),

        F.count(
            F.when(
                (F.col("event") == "transaction")
                & (F.col("time_since_test_start") <= 168),
                1
            )
        ).alias("transactions_first_week"),

        F.count(
            F.when(
                (F.col("event") == "transaction")
                & (F.col("time_since_test_start") > 168)
                & (F.col("time_since_test_start") <= 336),
                1
            )
        ).alias("transactions_second_week")
    )
)

time_features.display()

In [0]:
from pyspark.sql import functions as F

# Create the offer_stats CTE
offer_stats = (
    df.alias("t")
    .filter(
        F.col("offer_id").isNotNull()
        & F.col("offer_type").isNotNull()
    )
    .groupBy("account_id", "offer_type")
    .agg(
        F.count(
            F.when(F.col("event") == "offer received", 1)
        ).alias("received"),

        F.count(
            F.when(F.col("event") == "offer viewed", 1)
        ).alias("viewed"),

        F.count(
            F.when(F.col("event") == "offer completed", 1)
        ).alias("completed")
    )
)

# Compute the final features
offer_type_features = (
    offer_stats
    .groupBy("account_id")
    .agg(
        F.max(
            F.when(
                (F.col("offer_type") == "bogo")
                & (F.col("received") > 0),
                F.col("completed") / F.col("received")
            ).otherwise(0)
        ).alias("bogo_completion_rate"),

        F.max(
            F.when(
                (F.col("offer_type") == "discount")
                & (F.col("received") > 0),
                F.col("completed") / F.col("received")
            ).otherwise(0)
        ).alias("discount_completion_rate"),

        F.max(
            F.when(
                (F.col("offer_type") == "informational")
                & (F.col("received") > 0),
                F.col("viewed") / F.col("received")
            ).otherwise(0)
        ).alias("informational_view_rate")
    )
)

offer_type_features.display()

In [0]:
from pyspark.sql import functions as F

# Create the channel_stats CTE
channel_stats = (
    df.alias("t")
    .filter(
        F.col("offer_id").isNotNull()
        & F.col("channels").isNotNull()
    )
    .groupBy("account_id", "channels")
    .agg(
        F.count(
            F.when(F.col("event") == "offer received", 1)
        ).alias("received"),
        F.count(
            F.when(F.col("event") == "offer completed", 1)
        ).alias("completed")
    )
)

# Compute the engagement features
channel_features = (
    channel_stats
    .groupBy("account_id")
    .agg(
        F.max(
            F.when(
                (F.col("channels") == "email")
                & (F.col("received") > 0),
                F.col("completed") / F.col("received")
            ).otherwise(0)
        ).alias("email_engagement"),

        F.max(
            F.when(
                (F.col("channels") == "mobile")
                & (F.col("received") > 0),
                F.col("completed") / F.col("received")
            ).otherwise(0)
        ).alias("mobile_engagement"),

        F.max(
            F.when(
                (F.col("channels") == "social")
                & (F.col("received") > 0),
                F.col("completed") / F.col("received")
            ).otherwise(0)
        ).alias("social_engagement"),

        F.max(
            F.when(
                (F.col("channels") == "web")
                & (F.col("received") > 0),
                F.col("completed") / F.col("received")
            ).otherwise(0)
        ).alias("web_engagement")
    )
)

channel_features.show()

In [0]:
transactions_df = spark.table(TABLE_CONFIG.TRASACTIONS)
offers_df = spark.table(TABLE_CONFIG.OFFERS)
profile_df = spark.table(TABLE_CONFIG.PROFILE)
df = spark.table(TABLE_CONFIG.DATA_PROCESSING)

In [0]:
offer_events = transactions_df \
    .filter(F.col("offer_id").isNotNull()) \
    .groupBy("account_id", "offer_id") \
    .agg(
        F.sum(F.when(F.col("event") == "offer received", 1).otherwise(0)).alias("cnt_received"),
        F.sum(F.when(F.col("event") == "offer viewed", 1).otherwise(0)).alias("cnt_viewed"),
        F.sum(F.when(F.col("event") == "offer completed", 1).otherwise(0)).alias("cnt_completed")
    ) \
    .withColumn("is_converted_valid", 
        F.when((F.col("cnt_viewed") > 0) & (F.col("cnt_completed") > 0), 1).otherwise(0)
    )
offer_events.display()

In [0]:
(channel_features
 .join(offer_type_features, on=['account_id'])
 .join(time_features, on=['account_id'])
 .join(reward_features, on=['account_id'])
 .join(transaction_features, on=['account_id'])
 .join(offer_features, on=['account_id'])).display()